# Vertex AI Custom Training — YOLO Object Detection

This notebook submits a YOLO training job to **Vertex AI Custom Training** using the project's existing Docker image.

**Flow:**
1. Install & configure GCP SDK
2. Build and push Docker image to Artifact Registry via Cloud Build
3. Upload dataset to GCS
4. Submit Custom Training Job
5. Monitor job and download results

**Prerequisites:**
- A GCP project with Vertex AI API enabled
- Artifact Registry API enabled
- Cloud Build API enabled
- A GCS bucket for data and model output

In [9]:
import os

REPO_URL = 'https://github.com/entz77/vizon_training.git'
REPO_BRANCH = 'macos_train'
REPO_DIR = '/content/vizon_training'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
    print(f'✓ Repo cloned ({REPO_BRANCH}) to {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch origin {REPO_BRANCH} && git -C {REPO_DIR} checkout {REPO_BRANCH} && git -C {REPO_DIR} pull')
    print(f'✓ Repo updated to latest {REPO_BRANCH}')

os.chdir(REPO_DIR)
print(f'✓ Working directory: {os.getcwd()}')

✓ Repo updated to latest macos_train
✓ Working directory: /content/vizon_training


## 1. Install Dependencies

In [10]:
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'google-cloud-aiplatform>=1.38.0',
    'google-cloud-storage>=2.10.0',
])
print('✓ Dependencies installed')

✓ Dependencies installed


## 2. Configuration

Set all project-specific values here. Everything below reads from these variables.

In [11]:
# ── GCP project ──────────────────────────────────────────────
PROJECT_ID   = 'vveesee'
REGION       = 'asia-southeast1'       # Change to your preferred region
BUCKET_NAME  = 'dev-vveesee-ml-training' # Must already exist

# ── Artifact Registry ────────────────────────────────────────
AR_REPO      = 'vizon-training'        # Artifact Registry repository name
IMAGE_NAME   = 'yolo-trainer'
IMAGE_TAG    = 'latest'

# ── Derived URIs (no edit needed) ────────────────────────────
AR_HOST          = f'{REGION}-docker.pkg.dev'
TRAINING_IMAGE   = f'{AR_HOST}/{PROJECT_ID}/{AR_REPO}/{IMAGE_NAME}:{IMAGE_TAG}'
GCS_ROOT         = f'gs://{BUCKET_NAME}/vizon-training'
GCS_DATA_DIR     = f'{GCS_ROOT}/datasets/rtd'
GCS_OUTPUT_DIR   = f'{GCS_ROOT}/runs'

# ── Training hyperparameters ─────────────────────────────────
DATASET_YAML     = 'configs/dataset_config.yaml'  # Local path; uploaded to GCS below
LOCAL_DATASET_DIR = 'datasets/rtd/'                     # Local dataset root to upload
MODEL_SIZE       = 'm'      # n | s | m | l | x
EPOCHS           = 100
BATCH_SIZE       = 16
IMGSZ            = 640
PATIENCE         = 20

# ── Vertex AI machine config ──────────────────────────────────
MACHINE_TYPE      = 'n1-standard-8'
ACCELERATOR_TYPE  = 'NVIDIA_TESLA_T4'  # or NVIDIA_TESLA_V100, NVIDIA_A100_80GB
ACCELERATOR_COUNT = 1

print(f'Project  : {PROJECT_ID}')
print(f'Region   : {REGION}')
print(f'Image    : {TRAINING_IMAGE}')
print(f'GCS root : {GCS_ROOT}')
print(f'Data Dir  : {GCS_DATA_DIR}')

Project  : vveesee
Region   : asia-southeast1
Image    : asia-southeast1-docker.pkg.dev/vveesee/vizon-training/yolo-trainer:latest
GCS root : gs://dev-vveesee-ml-training/vizon-training
Data Dir  : gs://dev-vveesee-ml-training/vizon-training/datasets/rtd


## 3. Authenticate & Initialise Vertex AI

In [12]:
import google.auth
from google.cloud import aiplatform

# Authenticate in Colab
from google.colab import auth
auth.authenticate_user()

credentials, project = google.auth.default()

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=GCS_ROOT,
)

print(f'✓ Authenticated as project: {project}')
print(f'✓ Vertex AI initialised — region: {REGION}')

✓ Authenticated as project: 
✓ Vertex AI initialised — region: asia-southeast1


## 4. Build & Push Docker Image via Cloud Build

Uses the project's `Dockerfile`. **Cloud Build** runs the build remotely — no local Docker daemon needed.

In [13]:
import subprocess

def run(cmd, **kwargs):
    """Run a shell command, stream stdout, and always surface stderr."""
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kwargs)
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, result.stdout, result.stderr)
    return result

# Ensure Cloud Build and Artifact Registry APIs are enabled
run(f'gcloud services enable cloudbuild.googleapis.com artifactregistry.googleapis.com --project {PROJECT_ID}')

# Create Artifact Registry repository (idempotent)
run(
    f'gcloud artifacts repositories create {AR_REPO} '
    f'--repository-format=docker '
    f'--location={REGION} '
    f'--project={PROJECT_ID} '
    f'--quiet || true'
)
print('✓ Artifact Registry repository ready')

$ gcloud services enable cloudbuild.googleapis.com artifactregistry.googleapis.com --project vveesee
Operation "operations/acat.p2-655525621458-17a43636-5ddb-4563-8e35-40148e3fbfb8" finished successfully.
$ gcloud artifacts repositories create vizon-training --repository-format=docker --location=asia-southeast1 --project=vveesee --quiet || true
ERROR: (gcloud.artifacts.repositories.create) ALREADY_EXISTS: the repository already exists
✓ Artifact Registry repository ready


In [14]:
# Cloud Build uploads the current directory as the build context,
# builds the Dockerfile remotely, and pushes the image to Artifact Registry.
run(
    f'gcloud builds submit . '
    f'--tag {TRAINING_IMAGE} '
    f'--project {PROJECT_ID} '
    f'--timeout=1200'
)

print(f'\n✓ Image built and pushed: {TRAINING_IMAGE}')

$ gcloud builds submit . --tag asia-southeast1-docker.pkg.dev/vveesee/vizon-training/yolo-trainer:latest --project vveesee --timeout=1200
----------------------------- REMOTE BUILD OUTPUT ------------------------------
starting build "cf4da2dc-2214-4636-87bb-999ce1b48705"

FETCHSOURCE
Fetching storage object: gs://vveesee_cloudbuild/source/1779121079.628112-c1ef0259b8394a6ea79b6504f8d5ed7f.tgz#1779121080477654
Copying gs://vveesee_cloudbuild/source/1779121079.628112-c1ef0259b8394a6ea79b6504f8d5ed7f.tgz#1779121080477654...
/ [0 files][    0.0 B/  1.2 MiB]                                                
/ [1 files][  1.2 MiB/  1.2 MiB]                                                
Operation completed over 1 objects/1.2 MiB.                                      
BUILD
Already have image (with digest): gcr.io/cloud-builders/gcb-internal
Sending build context to Docker daemon  161.3kB

Step 1/11 : FROM pytorch/pytorch:2.0.0-cuda11.7-cudnn8-runtime
2.0.0-cuda11.7-cudnn8-runtime: Pulling fr

## 5. Create Dataset Config in GCS

Dataset must be uploaded to GCS **from your local machine** before running this notebook:
```bash
gsutil -m cp -r datasets/rtd/ gs://dev-vveesee-ml-training/vizon-training/datasets/rtd/
```

In [15]:
import yaml
from google.cloud import storage

# Load local dataset config and rewrite paths to point at the GCS download
# The training container downloads the dataset to /tmp/dataset at startup.
CONTAINER_DATA_DIR = '/tmp/dataset'
gcs_dataset_prefix = 'vizon-training/datasets/rtd'

with open(DATASET_YAML) as f:
    ds_cfg = yaml.safe_load(f)

gcs_dataset_yaml_cfg = dict(ds_cfg)
gcs_dataset_yaml_cfg['path'] = CONTAINER_DATA_DIR

GCS_DATASET_YAML_LOCAL = '/tmp/gcs_dataset_config.yaml'
with open(GCS_DATASET_YAML_LOCAL, 'w') as f:
    yaml.dump(gcs_dataset_yaml_cfg, f)

# Upload the patched YAML
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)
yaml_blob_name = f'{gcs_dataset_prefix}/gcs_dataset_config.yaml'
bucket.blob(yaml_blob_name).upload_from_filename(GCS_DATASET_YAML_LOCAL)
GCS_DATASET_YAML_URI = f'gs://{BUCKET_NAME}/{yaml_blob_name}'

print(f'✓ GCS dataset YAML uploaded: {GCS_DATASET_YAML_URI}')
print(f'  Container data path: {CONTAINER_DATA_DIR}')
print(f'  Classes: {ds_cfg.get("names")}')

✓ GCS dataset YAML uploaded: gs://dev-vveesee-ml-training/vizon-training/datasets/rtd/gcs_dataset_config.yaml
  Container data path: /tmp/dataset
  Classes: {0: 'RTD', 1: 'Shelf'}


## 6. Submit Vertex AI Custom Training Job

The container's entrypoint is overridden with a one-liner that:
1. Downloads the dataset from GCS → `/tmp/dataset`
2. Downloads the dataset YAML from GCS
3. Runs `train.py`
4. Copies results to `$AIP_MODEL_DIR` (Vertex AI's managed output path)

In [16]:
from datetime import datetime

JOB_NAME = f'vizon-yolo-{MODEL_SIZE}-{datetime.now().strftime("%Y%m%d-%H%M%S")}'

TRAIN_COMMAND = (
    f'mkdir -p {CONTAINER_DATA_DIR} && '
    f'gsutil -m cp -r {GCS_DATA_DIR} {CONTAINER_DATA_DIR} && '
    f'gsutil cp {GCS_DATASET_YAML_URI} /tmp/dataset_config.yaml && '
    f'python train.py '
    f'--data /tmp/dataset_config.yaml '
    f'--config configs/training_config.yaml '
    f'--epochs {EPOCHS} '
    f'--batch-size {BATCH_SIZE} '
    f'--imgsz {IMGSZ} '
    f'--patience {PATIENCE} '
    f'--save-dir /tmp/runs '
    f'--log-dir /tmp/logs '
    f'--use-cuda '
    f'--validate && '
    f'gsutil -m cp -r /tmp/runs $AIP_MODEL_DIR && '
    f'gsutil -m cp -r /tmp/logs $AIP_MODEL_DIR'
)

job = aiplatform.CustomContainerTrainingJob(
    display_name=JOB_NAME,
    container_uri=TRAINING_IMAGE,
    command=['bash', '-c', TRAIN_COMMAND],
)

print(f'Job name : {JOB_NAME}')
print(f'Image    : {TRAINING_IMAGE}')
print(f'Machine  : {MACHINE_TYPE} + {ACCELERATOR_COUNT}x {ACCELERATOR_TYPE}')
print(f'Output   : {GCS_OUTPUT_DIR}')
print('\nSubmitting job…')

job.run(
    base_output_dir=GCS_OUTPUT_DIR,
    replica_count=1,
    machine_type=MACHINE_TYPE,
    accelerator_type=ACCELERATOR_TYPE,
    accelerator_count=ACCELERATOR_COUNT,
    sync=False,
)

print(f'\n✓ Job submitted: {JOB_NAME}')
print(f'  Console: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project={PROJECT_ID}')

Job name : vizon-yolo-m-20260518-162330
Image    : asia-southeast1-docker.pkg.dev/vveesee/vizon-training/yolo-trainer:latest
Machine  : n1-standard-8 + 1x NVIDIA_TESLA_T4
Output   : gs://dev-vveesee-ml-training/vizon-training/runs

Submitting job…

✓ Job submitted: vizon-yolo-m-20260518-162330
  Console: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project=vveesee


## 7. Monitor Job

In [ ]:
import time
from google.cloud import aiplatform as aip

print(f'Monitoring: {JOB_NAME}')
print(f'Console: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project={PROJECT_ID}\n')

# Wait for job to be registered before polling
time.sleep(10)

TERMINAL_STATES = {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'}

while True:
    jobs = aip.CustomContainerTrainingJob.list(
        filter=f'display_name="{JOB_NAME}"',
        project=PROJECT_ID,
        location=REGION,
    )
    if not jobs:
        print('Job not found yet, waiting…')
        time.sleep(15)
        continue

    current_job = jobs[0]
    state = current_job.state.name
    print(f'[{datetime.now().strftime("%H:%M:%S")}] State: {state}', flush=True)
    if state in TERMINAL_STATES:
        break
    time.sleep(60)

if state == 'JOB_STATE_SUCCEEDED':
    print(f'\n✓ Training completed successfully')
    print(f'  Output: {GCS_OUTPUT_DIR}')
else:
    print(f'\n✗ Job ended with state: {state}')
    print('  Check Vertex AI console for logs.')

## 8. Download Results from GCS

In [ ]:
from pathlib import Path
from google.cloud import storage

LOCAL_RESULTS_DIR = Path('vertex_results') / JOB_NAME
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

gcs_output_prefix = GCS_OUTPUT_DIR.replace(f'gs://{BUCKET_NAME}/', '')
blobs = list(bucket.list_blobs(prefix=gcs_output_prefix))

downloaded = 0
for blob in blobs:
    relative = blob.name[len(gcs_output_prefix):].lstrip('/')
    if not relative:
        continue
    dest = LOCAL_RESULTS_DIR / relative
    dest.parent.mkdir(parents=True, exist_ok=True)
    blob.download_to_filename(str(dest))
    downloaded += 1

print(f'✓ Downloaded {downloaded} files to {LOCAL_RESULTS_DIR}')

In [ ]:
import glob

# Find best weights
weights = sorted(glob.glob(str(LOCAL_RESULTS_DIR / '**' / 'best.pt'), recursive=True))
if weights:
    print('Best weights:')
    for w in weights:
        size_mb = Path(w).stat().st_size / 1e6
        print(f'  {w}  ({size_mb:.1f} MB)')
else:
    print('No best.pt found — check if job succeeded.')

# Find training logs
logs = sorted(glob.glob(str(LOCAL_RESULTS_DIR / '**' / '*.log'), recursive=True))
if logs:
    print(f'\nLogs ({len(logs)} files):')
    for l in logs:
        print(f'  {l}')

## 9. (Optional) Run Inference on Downloaded Weights

Verify the downloaded model works locally before deploying.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))

from src.models import YOLOModel
from src.inference import YOLOPredictor

best_weight = weights[0] if weights else None

if best_weight:
    model = YOLOModel(model_size=MODEL_SIZE, device='cpu')
    model.load_weights(best_weight)

    predictor = YOLOPredictor(model, conf_threshold=0.5, iou_threshold=0.45)

    # Replace with an actual test image path
    TEST_IMAGE = 'datasets/homecare/images/test/sample.jpg'
    if Path(TEST_IMAGE).exists():
        results = predictor.predict_image(TEST_IMAGE)
        print(f'Detections on {TEST_IMAGE}:')
        for det in results['detections']:
            print(f"  {det['class_name']} — conf: {det['confidence']:.2f}")
    else:
        print(f'Test image not found at {TEST_IMAGE}')
        print('Set TEST_IMAGE to a valid path to run inference.')
else:
    print('No weights downloaded — skipping inference check.')